In [ ]:
# 02 — Build the training windows. Turns the flat feature table into the fixed
# 48-hour input windows the models consume, each paired with the PM2.5 value 24
# hours later. Splits them by time (70/15/15), fits the feature scaler on the
# TRAINING data only (to avoid leakage), and saves everything the later
# notebooks need.
# Input: feat_airquality.parquet
# Output: artifacts/windows.npz (the split, scaled windows) + preprocess.joblib (scaler + config)

In [2]:
import sys; from pathlib import Path
ROOT = Path.cwd(); ROOT = ROOT if (ROOT/"common").exists() else ROOT.parent
sys.path.insert(0, str(ROOT/"modeling"))
import numpy as np, pandas as pd, joblib
from windowing import build_supervised, FEATURE_COLS, SEQ_LEN, HORIZON
from splits import time_split, fit_feature_scaler, apply_scaler

df = pd.read_parquet(ROOT/"modeling"/"feat_airquality.parquet")
X, y, meta = build_supervised(df)                       # FEATURE_COLS by default
tr, va, te = time_split(meta)

sc = fit_feature_scaler(X[tr])                          # TRAIN ONLY
Xtr, Xva, Xte = apply_scaler(sc, X[tr]), apply_scaler(sc, X[va]), apply_scaler(sc, X[te])

assert meta["valid_time"].iloc[tr].max() <= meta["valid_time"].iloc[va].min()   # no leakage
print("X:", X.shape, "| features:", len(FEATURE_COLS))
print("train/val/test:", len(tr), len(va), len(te))

out = ROOT/"modeling"/"artifacts"; out.mkdir(exist_ok=True)
np.savez(out/"windows.npz", Xtr=Xtr, ytr=y[tr], Xva=Xva, yva=y[va], Xte=Xte, yte=y[te])
joblib.dump({"scaler": sc, "feature_cols": FEATURE_COLS, "seq_len": SEQ_LEN, "horizon": HORIZON},
            out/"preprocess.joblib")
print("saved -> modeling/artifacts/")

X: (63942, 48, 13) | features: 13
train/val/test: 44760 9591 9591
saved -> modeling/artifacts/
